# Prepare Face Attribute Dataset

Build a full-size CelebA-derived face attribute dataset for the recommendation bridge, using clear interpretable fields derived from native CelebA labels.

In [1]:
from pathlib import Path
import sys

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'backend').exists() and (candidate / 'notebooks').exists():
            return candidate
    raise RuntimeError('Could not locate project root.')

PROJECT_ROOT = find_project_root()
BACKEND_ROOT = PROJECT_ROOT / 'backend'
if str(BACKEND_ROOT) not in sys.path:
    sys.path.append(str(BACKEND_ROOT))

PROJECT_ROOT

WindowsPath('.')

In [2]:
import json
import pandas as pd

from systems.static_auto_tryon.auto_app.ml.datasets import build_label_vocab_for_fields, write_jsonl_manifest

RAW_CELEBA_ROOT = BACKEND_ROOT / 'data' / 'raw' / 'celeba'
IMAGE_DIR_CANDIDATES = [
    RAW_CELEBA_ROOT / 'img_align_celeba' / 'img_align_celeba',
    RAW_CELEBA_ROOT / 'img_align_celeba',
]
IMAGE_ROOT = next((path for path in IMAGE_DIR_CANDIDATES if path.exists()), None)
if IMAGE_ROOT is None:
    raise FileNotFoundError('Could not locate the CelebA image directory.')

ATTR_PATH = RAW_CELEBA_ROOT / 'list_attr_celeba.csv'
PARTITION_PATH = RAW_CELEBA_ROOT / 'list_eval_partition.csv'

DATASET_ROOT = BACKEND_ROOT / 'data' / 'datasets' / 'celeba_face_strong'
TRAIN_PATH = DATASET_ROOT / 'train.jsonl'
VAL_PATH = DATASET_ROOT / 'val.jsonl'
TEST_PATH = DATASET_ROOT / 'test.jsonl'
VOCAB_PATH = DATASET_ROOT / 'label_vocab.json'
SUMMARY_PATH = DATASET_ROOT / 'summary.json'

FACE_FIELDS = {
    'gender': ('Male', {1: 'male', -1: 'female'}),
    'face_fullness': ('Chubby', {1: 'full', -1: 'slim'}),
    'cheekbones': ('High_Cheekbones', {1: 'high', -1: 'soft'}),
    'hairline': ('Receding_Hairline', {1: 'receding', -1: 'regular'}),
}

LIGHTWEIGHT_MODE = False
MAX_TRAIN = 2000 if LIGHTWEIGHT_MODE else None
MAX_VAL = 400 if LIGHTWEIGHT_MODE else None
MAX_TEST = 400 if LIGHTWEIGHT_MODE else None

DATASET_ROOT

WindowsPath('./backend/data/datasets/celeba_face_strong')

In [3]:
attrs = pd.read_csv(ATTR_PATH)
partitions = pd.read_csv(PARTITION_PATH)
frame = attrs.merge(partitions, on='image_id', how='inner')

filtered = frame[(frame['Wearing_Hat'] != 1) & (frame['Bald'] != 1)].copy()
filtered['partition_name'] = filtered['partition'].map({0: 'train', 1: 'val', 2: 'test'})

print('Merged rows:', len(frame))
print('Filtered rows:', len(filtered))
filtered[['image_id', 'partition_name'] + [column for column, _ in FACE_FIELDS.values()]].head(10)

Merged rows: 202599
Filtered rows: 188257


,image_id,partition_name,Male,Chubby,High_Cheekbones,Receding_Hairline
0,000001.jpg,train,-1,-1,1,-1
1,000002.jpg,train,-1,-1,1,-1
2,000003.jpg,train,1,-1,-1,-1
3,000004.jpg,train,-1,-1,-1,-1
4,000005.jpg,train,-1,-1,-1,-1
5,000006.jpg,train,-1,-1,-1,-1
6,000007.jpg,train,1,-1,-1,-1
7,000008.jpg,train,1,-1,-1,-1
8,000009.jpg,train,-1,-1,1,-1
9,000010.jpg,train,-1,-1,1,-1


In [4]:
limits = {'train': MAX_TRAIN, 'val': MAX_VAL, 'test': MAX_TEST}
records = []

for partition_name, limit in limits.items():
    partition_rows = filtered[filtered['partition_name'] == partition_name].copy()
    if limit is not None:
        partition_rows = partition_rows.head(limit)

    for _, row in partition_rows.iterrows():
        labels = {
            field: mapping[int(row[source_column])]
            for field, (source_column, mapping) in FACE_FIELDS.items()
        }
        records.append({
            'image_path': str(IMAGE_ROOT / row['image_id']),
            'labels': labels,
            'source_id': row['image_id'],
            'source_dataset': 'CelebA-face-basic',
            'partition': partition_name,
        })

print('Prepared records:', len(records))
records[0] if records else None

Prepared records: 188257


{'image_path': 'backend/data\\raw\\celeba\\img_align_celeba\\img_align_celeba\\000001.jpg',
 'labels': {'gender': 'female',
  'face_fullness': 'slim',
  'cheekbones': 'high',
  'hairline': 'regular'},
 'source_id': '000001.jpg',
 'source_dataset': 'CelebA-face-basic',
 'partition': 'train'}

In [5]:
train_records = [record for record in records if record['partition'] == 'train']
val_records = [record for record in records if record['partition'] == 'val']
test_records = [record for record in records if record['partition'] == 'test']

label_vocab = build_label_vocab_for_fields(records, FACE_FIELDS.keys())

DATASET_ROOT.mkdir(parents=True, exist_ok=True)
write_jsonl_manifest(train_records, TRAIN_PATH)
write_jsonl_manifest(val_records, VAL_PATH)
write_jsonl_manifest(test_records, TEST_PATH)
VOCAB_PATH.write_text(json.dumps(label_vocab, indent=2), encoding='utf-8')

summary = {
    'face_fields': list(FACE_FIELDS.keys()),
    'total_records': len(records),
    'train_records': len(train_records),
    'val_records': len(val_records),
    'test_records': len(test_records),
    'field_value_counts': {
        field: pd.Series([record['labels'][field] for record in records]).value_counts().to_dict()
        for field in FACE_FIELDS.keys()
    },
}
SUMMARY_PATH.write_text(json.dumps(summary, indent=2), encoding='utf-8')

print('Wrote:', TRAIN_PATH)
print('Wrote:', VAL_PATH)
print('Wrote:', TEST_PATH)
print('Wrote:', VOCAB_PATH)
print('Wrote:', SUMMARY_PATH)

Wrote: backend/data\datasets\celeba_face_strong\train.jsonl
Wrote: backend/data\datasets\celeba_face_strong\val.jsonl
Wrote: backend/data\datasets\celeba_face_strong\test.jsonl
Wrote: backend/data\datasets\celeba_face_strong\label_vocab.json
Wrote: backend/data\datasets\celeba_face_strong\summary.json


In [6]:
pd.Series(summary)

face_fields               [gender, face_fullness, cheekbones, hairline]
total_records                                                    188257
train_records                                                    151039
val_records                                                       18517
test_records                                                      18701
field_value_counts    {'gender': {'female': 115201, 'male': 73056}, ...
dtype: object